# 99 — Validation inter-étapes et invariants

Ce notebook bloque les régressions structurelles et métier sur l'ensemble du pipeline
`00 → 01 → 02 → 03 → 04 → 05 → 06` : cardinalité, unicité de `stage_id`, refus de canoniser les
divergences réelles, pays ISO-2, confiance des inférences, statuts codes postaux cohérents,
absence des anciennes colonnes ambiguës, intégrité du dataset final (unique producteur : l'étape
05) et hygiène du dépôt (pas de livrables dupliqués ou orphelins).

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from stages_cleaning.identity import SAFE_CANONICALIZATION_CATEGORIES
from stages_cleaning.normalize import is_iso_alpha2
from stages_cleaning.postal import POSTAL_CODE_PATTERNS, validate_postal_code
from stages_cleaning.provenance import build_stage_id
from stages_cleaning.paths import dataset_paths

SOURCES = {
    "00_raw": (dataset_paths(ROOT).raw, ";"),
    "01_with_name": (ROOT / "data" / "interim" / "stages_efrei_with_name.csv", ","),
    "02_with_country": (ROOT / "data" / "interim" / "stages_efrei_with_country.csv", ","),
    "03_with_postal_code": (ROOT / "data" / "interim" / "stages_efrei_with_postal_code.csv", ","),
    "04_with_programme": (ROOT / "data" / "interim" / "stages_efrei_with_programme.csv", ","),
    "05_final": (ROOT / "data" / "processed" / "stages_efrei_final.csv", ","),
}
datasets = {name: pd.read_csv(path, sep=sep, dtype="string", encoding="utf-8-sig", keep_default_na=False).replace("", pd.NA) for name, (path, sep) in SOURCES.items()}
print({k: v.shape for k, v in datasets.items()})

## 1. Cardinalité et identifiants

In [ ]:
raw = datasets["00_raw"]
raw_ids = build_stage_id(raw)
source_cols = list(raw.columns)
expected = len(raw) - raw.duplicated(subset=source_cols).sum()
stages_after_dedup = ["01_with_name", "02_with_country", "03_with_postal_code", "04_with_programme", "05_final"]
for stage in stages_after_dedup:
    frame = datasets[stage]
    assert len(frame) == expected, f"Cardinalité inattendue pour {stage}: {len(frame)} != {expected}"
    assert frame["stage_id"].is_unique, f"stage_id non unique dans {stage}"
expected_ids = set(raw_ids)
for stage in stages_after_dedup:
    assert set(datasets[stage]["stage_id"]) == expected_ids, f"stage_id perdu ou ajouté dans {stage}"
print("✔ Cardinalité et stage_id cohérents sur tout le pipeline")

## 1bis. Non-régression des colonnes héritées (01→02, 02→03)

Complète les comparaisons 03→04 et 04→05 déjà présentes plus bas : chaque étape ne doit modifier que les colonnes qu'elle a explicitement le droit de réécrire (déclarées en positif ci-dessous), le reste doit être identique à l'étape précédente.


In [ ]:
# Non-régression des colonnes héritées sur TOUT le pipeline (contrat README).
named   = datasets["01_with_name"].sort_values("stage_id").reset_index(drop=True)
country = datasets["02_with_country"].sort_values("stage_id").reset_index(drop=True)
postal  = datasets["03_with_postal_code"].sort_values("stage_id").reset_index(drop=True)

MUTABLE_02 = {"ville", "pays", "code_postal"}   # colonnes que l'étape 02 réécrit légitimement
MUTABLE_03 = {"code_postal"}                     # idem étape 03

inherited_01_02 = [c for c in named.columns if c in country.columns and c not in MUTABLE_02]
pd.testing.assert_frame_equal(named[inherited_01_02], country[inherited_01_02], check_dtype=False)

inherited_02_03 = [c for c in country.columns if c in postal.columns and c not in MUTABLE_03]
pd.testing.assert_frame_equal(country[inherited_02_03], postal[inherited_02_03], check_dtype=False)
print("Colonnes héritées 01→02 et 02→03 : inchangées.")


## 2. Identités : canonisation conservatrice

In [ ]:
named = datasets["01_with_name"]
unsafe = named["identity_variant_category"].eq("divergence_reelle")
assert not named.loc[unsafe, "identity_canonicalized"].eq("True").any()
assert named.loc[unsafe, "identity_requires_review"].eq("True").all()
safe = named["identity_variant_category"].isin(SAFE_CANONICALIZATION_CATEGORIES)
safe_graphies = named.loc[safe].groupby("email")[["nom", "prenom"]].nunique(dropna=False)
assert (safe_graphies <= 1).all().all(), "Une variation sûre n'a pas été harmonisée"
print("✔ Divergences réelles préservées et variations sûres harmonisées")

## 3. Pays et niveau de preuve

In [ ]:
country = datasets["02_with_country"]
non_iso = country["pays"].notna() & ~country["pays"].map(is_iso_alpha2)
assert not non_iso.any(), "Pays final non ISO-2"
assert not country["country_resolution_method"].eq("postal_code_unique_source").any()
assert country.loc[country["pays"].isna(), "country_resolution_method"].eq("unresolved").all()
assert country.loc[country["country_resolution_method"].isin(["postal_code_city", "city_unique_source"]), "country_reference_count"].notna().all()
assert country["country_resolution_confidence"].notna().all()
print("✔ Pays ISO-2, aucune inférence CP seul, confiance et support tracés")

## 4. Codes postaux : validité et sémantique des statuts

In [ ]:
postal = datasets["03_with_postal_code"]
checks = pd.Series([validate_postal_code(cp, country) for cp, country in zip(postal["code_postal"], postal["pays"])], index=postal.index, dtype="object")
valid_status = postal["postal_code_final_status"].eq("valid_country_pattern")
assert checks.loc[valid_status].eq(True).all()
unresolved = postal["postal_code_final_status"].eq("unresolved")
assert postal.loc[unresolved, "code_postal"].isna().all()
assert postal.loc[unresolved, "postal_code_resolution_method"].str.startswith("unresolved", na=False).all()
country_missing = postal["postal_code_final_status"].eq("unvalidated_country_missing")
assert postal.loc[country_missing, "pays"].isna().all()
unsupported = postal["postal_code_final_status"].eq("unvalidated_pattern_unsupported")
assert postal.loc[unsupported, "pays"].notna().all()
assert ~postal.loc[unsupported, "pays"].isin(POSTAL_CODE_PATTERNS).any()
placeholder_source = postal["postal_code_source_status"].eq("suspicious_placeholder")
assert not postal.loc[placeholder_source, "postal_code_resolution_method"].str.startswith("source", na=False).any()
print("✔ Statuts postaux cohérents et placeholders jamais acceptés comme source")

## 5. Intégrité de l'enrichissement programme (03 → 04)

In [ ]:
postal = datasets["03_with_postal_code"].sort_values("stage_id").reset_index(drop=True)
with_programme = datasets["04_with_programme"].sort_values("stage_id").reset_index(drop=True)
required_programme = {
    "stage_id", "real_program", "program_year", "major", "major_status", "programme_campus",
    "is_swim_program", "is_swim_stage", "is_international_stage",
    "country_resolution_confidence", "country_reference_count",
    "postal_code_final_status",
}
missing = required_programme - set(with_programme.columns)
assert not missing, f"Colonnes programme manquantes : {sorted(missing)}"
for deprecated in ["is_swim", "campus", "swim_from_country"]:
    assert deprecated not in with_programme.columns, f"Colonne ambiguë réintroduite : {deprecated}"
pd.testing.assert_frame_equal(postal[postal.columns], with_programme[postal.columns], check_dtype=False)
assert with_programme.loc[with_programme["pays"].isna(), "is_international_stage"].isna().all()
expected_swim_stage = with_programme["type_stage"].str.strip().str.upper().eq("SWIM")
actual_swim_stage = with_programme["is_swim_stage"].map({"True": True, "False": False}).astype("boolean")
assert actual_swim_stage.equals(expected_swim_stage.astype("boolean"))
print("✔ L'étape 04 enrichit sans altérer les colonnes de l'étape 03")
print("✔ Les notions SWIM programme, SWIM stage et international sont séparées")

## 6. Intégrité de la normalisation entreprise (04 → 05, producteur unique du final)

In [ ]:
with_programme = datasets["04_with_programme"].sort_values("stage_id").reset_index(drop=True)
final = datasets["05_final"].sort_values("stage_id").reset_index(drop=True)
required_company = {"entreprise_normalisee", "entreprise_qualite", "entreprise_canonique", "entreprise_famille"}
missing_company = required_company - set(final.columns)
assert not missing_company, f"Colonnes entreprise manquantes dans le final : {sorted(missing_company)}"
pd.testing.assert_frame_equal(with_programme[with_programme.columns], final[with_programme.columns], check_dtype=False)
assert final["entreprise_canonique"].notna().equals(final["entreprise_normalisee"].notna()), "entreprise_canonique doit être renseignée exactement quand entreprise_normalisee l'est"
print("✔ L'étape 05 enrichit sans altérer les colonnes programme/pays/CP/identité")
print("✔ Le dataset final réunit identité, pays, code postal, programme et entreprise en un seul fichier")

## 7. Livrables et hygiène du dépôt

In [ ]:
business_report = ROOT / "data" / "reports" / "rapport_qualite_metier.xlsx"
technical_report = ROOT / "data" / "reports" / "rapport_qualite_technique.xlsx"
issue_registry = ROOT / "data" / "reports" / "quality_issues.csv"
pdf_summary = ROOT / "data" / "reports" / "resume_qualite.pdf"
for path in [business_report, technical_report, issue_registry, pdf_summary]:
    assert path.exists(), f"Livrable qualité manquant : {path}. Lancez `python generate_quality_reports.py` (ou le notebook 06)."

# Anciens fichiers de sortie divergents : ne doivent plus jamais réapparaître (garde contre la
# régression corrigée lors du refactor du 2026-07 : deux "finaux" concurrents + rapport orphelin).
deprecated_outputs = [
    ROOT / "data" / "processed" / "stages_efrei_clean.csv",
    ROOT / "data" / "processed" / "stages_efrei_companies_cleaned.csv",
    ROOT / "data" / "processed" / "rapport_qualite_donnees.xlsx",
]
for path in deprecated_outputs:
    assert not path.exists(), f"Fichier obsolète réapparu, à supprimer : {path}"

assert not (ROOT / "notebooks" / "Untitled.ipynb").exists(), "Untitled.ipynb doit être supprimé"
print("✔ Rapports qualité présents, aucun livrable dupliqué ou orphelin, aucun notebook parasite")